In [ ]:
import glob
import pickle
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict


# --------------------------------------------------
# USER INPUT
# --------------------------------------------------
TIMEAVG_GLOB = "/om2/user/bjmedina/auditory-memory/memory/fits/v13_three-regime_time_avg_all/sigma0/*experiments.pkl"
NOTIMEAVG_GLOB = "/om2/user/bjmedina/auditory-memory/memory/fits/v13_three-regime_nontime_avg_all/sigma0/*experiments.pkl"

# --------------------------------------------------
# Helper: load + merge dicts
# --------------------------------------------------
def load_and_merge(pkl_files, isi_idx=0):
    sigma0_by_n = {}
    mse_by_n = {}
    perf_by_n = defaultdict(list)

    for p in pkl_files:
        with open(p, "rb") as f:
            out = pickle.load(f)

        # σ0
        for n, vals in out["sigma0_by_n"].items():
            sigma0_by_n.setdefault(n, []).extend(vals)

        # MSE
        for n, vals in out["mse_by_n"].items():
            mse_by_n.setdefault(n, []).extend(vals)

        # PERFORMANCE (from best_models)
        for n, models in out["best_models"].items():
            for m in models:
                perf_by_n[n].append(m['model_dp'][isi_idx])

    return sigma0_by_n, mse_by_n, perf_by_n

timeavg_files = sorted(glob.glob(TIMEAVG_GLOB))
notimeavg_files = sorted(glob.glob(NOTIMEAVG_GLOB))


sigma0_time, mse_time, perf_time = load_and_merge(timeavg_files, isi_idx=0)
sigma0_notime, mse_notime, perf_notime = load_and_merge(notimeavg_files, isi_idx=0)
# --------------------------------------------------
# Plot: σ0 stability comparison
# --------------------------------------------------
plt.figure(figsize=(7, 4))

xs = sorted(set(sigma0_time) & set(sigma0_notime))

med_time = []
med_notime = []

for n in xs:
    sig_t = np.asarray(sigma0_time[n])
    sig_nt = np.asarray(sigma0_notime[n])

    plt.scatter(
        np.full_like(sig_t, n, dtype=float) - 0.05,
        sig_t,
        alpha=0.25,
        color="tab:blue"
    )
    plt.scatter(
        np.full_like(sig_nt, n, dtype=float) + 0.05,
        sig_nt,
        alpha=0.25,
        color="tab:orange"
    )

    med_time.append(np.median(sig_t))
    med_notime.append(np.median(sig_nt))

plt.plot(xs, med_time, "-o", color="tab:blue", label="time-avg")
plt.plot(xs, med_notime, "-o", color="tab:orange", label="no time-avg")

plt.xlabel("Subsample size")
plt.ylabel("Best-fit σ₀ (ISI=0)")
plt.title("σ₀ stability: time-averaged vs non–time-averaged")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

# -------------------------
# Time-averaged panel
# -------------------------
ax = axes[0]
xs_time = sorted(sigma0_time.keys())
med_time = []

for n in xs_time:
    sigs = np.asarray(sigma0_time[n])
    x = np.full(len(sigs), n)
    ax.scatter(x, sigs, alpha=0.3)
    med_time.append(np.median(sigs))
    ax.scatter(n, np.median(sigs), color="r", alpha=0.6)

ax.plot(xs_time, med_time, color="black", linewidth=2)
ax.set_title("Time-averaged")
ax.set_xlabel("Subsample size")
ax.set_ylabel("Best-fit σ₀ (ISI=0)")

# -------------------------
# Non–time-averaged panel
# -------------------------
ax = axes[1]
xs_notime = sorted(sigma0_notime.keys())
med_notime = []

for n in xs_notime:
    sigs = np.asarray(sigma0_notime[n])
    x = np.full(len(sigs), n)
    ax.scatter(x, sigs, alpha=0.3)
    med_notime.append(np.median(sigs))
    ax.scatter(n, np.median(sigs), color="r", alpha=0.6)

ax.plot(xs_notime, med_notime, color="black", linewidth=2)
ax.set_title("Non–time-averaged")
ax.set_xlabel("Subsample size")

plt.suptitle("σ₀ stability vs data size", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# -------------------------
# Time-averaged MSE panel
# -------------------------
ax = axes[0]
xs_time = sorted(mse_time.keys())
med_time = []

for n in xs_time:
    mses = np.asarray(mse_time[n])
    x = np.full(len(mses), n)
    ax.scatter(x, mses, alpha=0.3)
    med = np.median(mses)
    med_time.append(med)
    ax.scatter(n, med, color="r", alpha=0.6)

ax.plot(xs_time, med_time, color="black", linewidth=2)
ax.set_title("Time-averaged")
ax.set_xlabel("Subsample size")
ax.set_ylabel("MSE (ISI=0)")

# -------------------------
# Non–time-averaged MSE panel
# -------------------------
ax = axes[1]
xs_notime = sorted(mse_notime.keys())
med_notime = []

for n in xs_notime:
    mses = np.asarray(mse_notime[n])
    x = np.full(len(mses), n)
    ax.scatter(x, mses, alpha=0.3)
    med = np.median(mses)
    med_notime.append(med)
    ax.scatter(n, med, color="r", alpha=0.6)

ax.plot(xs_notime, med_notime, color="black", linewidth=2)
ax.set_title("Non–time-averaged")
ax.set_xlabel("Subsample size")

plt.suptitle("MSE stability vs data size", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
len(perf_time)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# -------------------------
# Time-averaged performance
# -------------------------
ax = axes[0]
xs_time = sorted(perf_time.keys())
med_time = []

for n in xs_time:
    vals = np.asarray(perf_time[n])
    x = np.full(len(vals), n)
    ax.scatter(x, vals, alpha=0.3)
    med = np.median(vals)
    med_time.append(med)
    ax.scatter(n, med, color="r", alpha=0.6)

ax.plot(xs_time, med_time, color="black", linewidth=2)
ax.set_title("Time-averaged")
ax.set_xlabel("Subsample size")
ax.set_ylabel("Performance (d′, ISI=0)")

# -------------------------
# Non–time-averaged performance
# -------------------------
ax = axes[1]
xs_notime = sorted(perf_notime.keys())
med_notime = []

for n in xs_notime:
    vals = np.asarray(perf_notime[n])
    x = np.full(len(vals), n)
    ax.scatter(x, vals, alpha=0.3)
    med = np.median(vals)
    med_notime.append(med)
    ax.scatter(n, med, color="r", alpha=0.6)

ax.plot(xs_notime, med_notime, color="black", linewidth=2)
ax.set_title("Non–time-averaged")
ax.set_xlabel("Subsample size")

plt.suptitle("Performance vs data size (d′ at ISI=0)", y=1.05)
plt.tight_layout()
plt.show()